<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/09_01_ggs416_26_03_30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️GGS416 Week 9: Plotting with Matplotlib🛰️

This week we will cover plotting options for data extracted from satellite imagery using `Matplotlib`.

This will include:

- Bar plots
- Stacked bar plots
- Example error bars (e.g., for the standard error / confidence intervals)
- Line graphs (e.g., for temporal detection change data)
- Additional graph types, including scatter plots, histograms, box plots and heatmaps

All datasets below are synthetic, which we generate, to avoid needing to get into data importing here.

I have designed them to resemble common remote sensing / Earth Observation (EO) outputs.

First, we need to load in some packages. You should be familiar with this already.

Here, we also set some parameters for the figure size, as well as title/label font size, and whether we will have a frame.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['legend.frameon'] = True

print('Libraries imported and plotting style configured.')

## Data generation

Imagine we sampled many image tiles and computed land cover class proportions in each tile (e.g., for 'Vegetation', 'Urban', 'Water') across multiple regions.

We create means, as well as uncertainty estimates to visualize in bar charts.


In [ ]:
regions = ['North', 'Central', 'South']
classes = ['Vegetation', 'Urban', 'Water']

# Simulated mean class composition (%) from sampled tiles
means = pd.DataFrame({
    'Vegetation': [58, 46, 63],
    'Urban': [30, 42, 22],
    'Water': [12, 12, 15]
}, index=regions)

# Simulated standard errors (%)
ses = pd.DataFrame({
    'Vegetation': [2.2, 2.8, 2.0],
    'Urban': [1.8, 2.1, 1.5],
    'Water': [1.1, 0.9, 1.2]
}, index=regions)

# Approximate 95% confidence intervals using 1.96 * SE
cis = 1.96 * ses

# Total mapped area by region (km²) to represent absolute magnitude
total_area_km2 = pd.Series({'North': 920, 'Central': 1380, 'South': 760})

# Convert composition (%) into absolute class area (km²)
class_area_km2 = means.mul(total_area_km2, axis=0) / 100

pd.concat(
    [
        means.add_suffix(' (%)'),
        class_area_km2.round(1).add_suffix(' area_km2')
    ],
    axis=1
)

## Grouped bar plot with error bars

Now we can create a grouped barplot to compare multiple classes across categories.

In [ ]:
x = np.arange(len(regions))
width = 0.24
colors = ['#2a9d8f', '#e76f51', '#457b9d']

fig, ax = plt.subplots(figsize=(11, 6))

for i, cls in enumerate(classes):
    offset = (i - 1) * width
    ax.bar(
        x + offset,
        means[cls],
        width=width,
        yerr=ses[cls],
        capsize=5,
        label=cls,
        color=colors[i],
        edgecolor='black',
        linewidth=0.6
    )

ax.set_xticks(x)
ax.set_xticklabels(regions)
ax.set_ylabel('Class Proportion (%)')
ax.set_title('Land-Cover Class Proportions by Region (with Standard Error)')
ax.legend(title='Class')
ax.set_ylim(0, 75)

plt.tight_layout()
plt.show()

## Stacked bar plot (composition + magnitude)

To show both composition and absolute magnitude, we can stack absolute class areas (km²).

- Total bar height = total mapped area in each region (magnitude)
- Each segment = class contribution within that total (composition)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(regions))
for i, cls in enumerate(classes):
    values = class_area_km2[cls].values
    bars = ax.bar(
        regions,
        values,
        bottom=bottom,
        label=cls,
        color=colors[i],
        edgecolor='white'
    )

    # Segment labels keep the composition visible on top of magnitude bars.
    for j, bar in enumerate(bars):
        pct = means.iloc[j, i]
        if values[j] > 80:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[j] + values[j] / 2,
                f'{pct:.0f}%',
                ha='center',
                va='center',
                color='white',
                fontsize=9,
                fontweight='bold'
            )

    bottom += values

# Annotate total magnitude above each stacked bar.
for i, region in enumerate(regions):
    total = total_area_km2[region]
    ax.text(i, total + 20, f'Total: {total:.0f} km²', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Mapped Area (km²)')
ax.set_title('Regional Land-Cover Composition and Absolute Area (Stacked Bar)')
ax.legend(title='Class', loc='upper right')
ax.set_ylim(0, total_area_km2.max() * 1.12)

plt.tight_layout()
plt.show()

## Time series line graphs

A common remote sensing task is tracking a vegetation index (e.g., NDVI) over time.

We can show both trend and uncertainty using a line plus a shaded confidence band.

In [ ]:
months = pd.date_range('2024-01-01', periods=12, freq='MS')
t = np.arange(12)

# Synthetic NDVI seasonal patterns
forest_mean = 0.55 + 0.18 * np.sin(2 * np.pi * (t - 2) / 12) + np.random.normal(0, 0.015, 12)
cropland_mean = 0.42 + 0.24 * np.sin(2 * np.pi * (t - 3) / 12) + np.random.normal(0, 0.02, 12)

forest_se = np.full(12, 0.015) + np.random.uniform(0, 0.004, 12)
crop_se = np.full(12, 0.02) + np.random.uniform(0, 0.005, 12)

forest_ci = 1.96 * forest_se
crop_ci = 1.96 * crop_se

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(months, forest_mean, marker='o', color='#2a9d8f', linewidth=2.2, label='Forest NDVI')
ax.fill_between(months, forest_mean - forest_ci, forest_mean + forest_ci, color='#2a9d8f', alpha=0.2)

ax.plot(months, cropland_mean, marker='s', color='#f4a261', linewidth=2.2, label='Cropland NDVI')
ax.fill_between(months, cropland_mean - crop_ci, cropland_mean + crop_ci, color='#f4a261', alpha=0.25)

ax.set_title('Seasonal NDVI Dynamics with 95% Confidence Intervals')
ax.set_ylabel('NDVI')
ax.set_xlabel('Month')
ax.legend()
ax.set_ylim(0.1, 0.85)

plt.tight_layout()
plt.show()

## Additional plot types

Below are a few more plot styles that are very common in satellite image analysis reporting.

In [ ]:
# Synthetic pixel-level sample for a scatter plot
n = 220
red = np.random.uniform(0.05, 0.45, n)
nir = 0.35 + 0.9 * red + np.random.normal(0, 0.06, n)
ndvi = (nir - red) / (nir + red)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(red, nir, c=ndvi, cmap='viridis', alpha=0.8, edgecolor='k', linewidth=0.2)

# Simple best-fit line
coef = np.polyfit(red, nir, 1)
fit_x = np.linspace(red.min(), red.max(), 100)
fit_y = coef[0] * fit_x + coef[1]
ax.plot(fit_x, fit_y, color='crimson', linewidth=2, label='Best-fit line')

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('NDVI')

ax.set_title('Scatter Plot: Red vs NIR Reflectance')
ax.set_xlabel('Red Reflectance')
ax.set_ylabel('NIR Reflectance')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Histogram: compare NDVI distributions from two dates
ndvi_spring = np.random.normal(0.58, 0.09, 400)
ndvi_summer = np.random.normal(0.66, 0.08, 400)

fig, ax = plt.subplots(figsize=(10, 6))
bins = np.linspace(0.2, 1.0, 28)
ax.hist(ndvi_spring, bins=bins, alpha=0.65, color='#457b9d', label='Spring', density=True)
ax.hist(ndvi_summer, bins=bins, alpha=0.55, color='#2a9d8f', label='Summer', density=True)

ax.set_title('Histogram: NDVI Distribution Shift by Season')
ax.set_xlabel('NDVI')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Box plot: cloud cover (%) by season for sampled scenes
cloud_data = [
    np.random.normal(42, 12, 60),  # Winter
    np.random.normal(35, 10, 60),  # Spring
    np.random.normal(22, 9, 60),   # Summer
    np.random.normal(30, 11, 60)   # Fall
]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(cloud_data, patch_artist=True, tick_labels=['Winter', 'Spring', 'Summer', 'Fall'])

box_colors = ['#6d597a', '#457b9d', '#2a9d8f', '#f4a261']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.65)

ax.set_title('Cloud Cover Variability by Season')
ax.set_ylabel('Cloud Cover (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: synthetic class confusion matrix (%)
labels = ['Vegetation', 'Urban', 'Water']
cm = np.array([
    [88, 9, 3],
    [11, 83, 6],
    [4, 7, 89]
])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='YlGnBu', vmin=0, vmax=100)

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted Class')
ax.set_ylabel('Reference Class')
ax.set_title('Classification Confusion Matrix (%)')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm[i, j]}', ha='center', va='center', color='black')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Percent (%)')

plt.tight_layout()
plt.show()

## Design tips for clear figures

Here are some basic tips to get you started with your report plots:

1. Use meaningful axis labels and units.
2. Prefer color palettes with good contrast and readability (which color blind folks can also see).
3. Show uncertainty (SE/CI) when reporting means (ideally).
4. Keep titles specific to the comparison you are making.
5. Use `plt.tight_layout()` to avoid clipped labels.

These habits will make your analysis figures easier to interpret, and also ensure you are meeting scientific best practise.

## Exercise

Create a new mini-analysis figure set using synthetic data that mimics one satellite workflow of your choice (e.g., burn severity, urban expansion, crop phenology, water extent).

Required outputs (ideally for your coursework topic of interest):

1. Build a stacked bar plot that shows composition + absolute magnitude.
2. Build one time series line plot with uncertainty shown as either error bars or a confidence band.
3. Build one additional plot type (scatter, histogram, box plot, or heatmap) that supports your story.

Requirements:

- Generate your own synthetic dataset in a code cell.
- Use clear axis labels with units.
- Use a consistent color palette across figures.
- Add figure titles that describe the analytical message.
- Include a short markdown interpretation explaining your key findings (300 words).


# Exercise

Now try have a go at researching and generating a panel plot.

For example, you could use a time series, like the months of the year, to try stitch together 12 individual plots (one for each month).